# RAG Evaluation

## Imports and Setup

In [6]:
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(".")

CHUNK_PATH     = BASE_DIR / "Results" / "chunk_data" / "rag_chunks.csv"
INDEX_PATH     = BASE_DIR / "Results" / "retriever_data" / "faiss.index"
LABELS_PATH    = BASE_DIR / "Results" / "retriever_data" / "starter_query_pairs_template_labelled.csv"

model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index(str(INDEX_PATH))
chunks = pd.read_csv(CHUNK_PATH)
labels = pd.read_csv(LABELS_PATH)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5559.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieve function

In [7]:
def retrieve(query, model, index, chunks, top_k = 5, filter_company = None, filter_quarter = None):
    # Embed the query
    q_emb = model.encode([query], convert_to_numpy = True)
    faiss.normalize_L2(q_emb)

    # Search — over-fetch to allow for filtering
    scores, indices = index.search(q_emb, top_k * 3)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks.iloc[idx]

        # Optional filters
        if filter_company and row["company"].lower() != filter_company.lower():
            continue
        if filter_quarter and row["quarter"] != filter_quarter:
            continue

        results.append({
            "chunk_id": row["chunk_id"],
            "company": row["company"],
            "quarter": row["quarter"],
            "score": float(score),
            "chunk_text": row["chunk_text"]
        })

        if len(results) == top_k:
            break

    return results

# Evaluation function


In [8]:
def evaluate_retriever(labels_df, top_k=5):
    precision_scores    = []
    recall_scores       = []
    reciprocal_ranks    = []
    hard_negative_wins  = []
    per_query_results   = []

    for _, row in labels_df.iterrows():
        query            = row["query"]
        company          = row["company"]
        quarter          = row["quarter"]
        positive_id      = row["positive_chunk_id"]
        hard_negative_id = row["hard_negative_chunk_id"]

        retrieved     = retrieve(query, model, index, chunks, top_k=top_k, filter_company=company, filter_quarter=quarter)
        retrieved_ids = [r["chunk_id"] for r in retrieved]

        # Precision@k
        hit            = 1 if positive_id in retrieved_ids else 0
        precision_at_k = hit / top_k
        precision_scores.append(precision_at_k)

        # Recall@k
        recall_scores.append(float(hit))

        # MRR
        if positive_id in retrieved_ids:
            rank = retrieved_ids.index(positive_id) + 1
            rr   = 1 / rank
        else:
            rank = None
            rr   = 0.0
        reciprocal_ranks.append(rr)

        if positive_id in retrieved_ids and hard_negative_id not in retrieved_ids:
            ranked_correctly = 1
        elif positive_id in retrieved_ids and hard_negative_id in retrieved_ids:
            pos_rank  = retrieved_ids.index(positive_id)
            hard_rank = retrieved_ids.index(hard_negative_id)
            ranked_correctly = 1 if pos_rank < hard_rank else 0
        else:
            ranked_correctly = 0

        hard_negative_wins.append(ranked_correctly)

        per_query_results.append({
            "company":                    company,
            "quarter":                    quarter,
            "query":                      query,
            "positive_id":                positive_id,
            "hard_negative_id":           hard_negative_id,
            "retrieved_ids":              retrieved_ids,
            "hit":                        hit,
            "rank":                       rank,
            "reciprocal_rank":            round(rr, 4),
            "ranked_above_hard_negative": ranked_correctly
        })

    metrics = {
        f"Precision@{top_k}":     round(np.mean(precision_scores), 4),
        f"Recall@{top_k}":        round(np.mean(recall_scores), 4),
        "MRR":                    round(np.mean(reciprocal_ranks), 4),
        "Hard negative accuracy": round(np.mean(hard_negative_wins), 4),
    }

    return metrics, pd.DataFrame(per_query_results)

## Evaluation 

In [9]:
k_values    = [1, 3, 5]
all_metrics = []

for k in k_values:
    metrics, _ = evaluate_retriever(labels, top_k=k)
    metrics["k"] = k
    all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics).set_index("k")
print(metrics_df.to_string())

   Precision@1  Recall@1     MRR  Hard negative accuracy  Precision@3  Recall@3  Precision@5  Recall@5
k                                                                                                     
1       0.0625    0.0625  0.0625                  0.0625          NaN       NaN          NaN       NaN
3          NaN       NaN  0.1589                  0.2031       0.0677    0.2031          NaN       NaN
5          NaN       NaN  0.1719                  0.2344          NaN       NaN       0.0469    0.2344


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, [f"Precision@k", f"Recall@k", "MRR"]):
    col = [c for c in metrics_df.columns if metric.split("@")[0] in c]
    if not col:
        col = ["MRR"]
    ax.plot(k_values, metrics_df[col[0]].values, marker="o")
    ax.set_title(col[0])
    ax.set_xlabel("k")
    ax.set_ylim(0, 1)
    ax.set_xticks(k_values)
    ax.grid(True, alpha=0.3)

plt.suptitle("Retriever performance across k", y=1.02)
plt.tight_layout()
plt.show()

In [11]:
_, results_df = evaluate_retriever(labels, top_k=5)

company_metrics = results_df.groupby("company").agg(
    hit_rate                = ("hit", "mean"),
    mrr                     = ("reciprocal_rank", "mean"),
    hard_negative_accuracy  = ("ranked_above_hard_negative", "mean")
).round(4)

print(company_metrics.to_string())

         hit_rate     mrr  hard_negative_accuracy
company                                          
amazon     0.3750  0.3438                  0.3750
apple      0.3125  0.1667                  0.3125
meta       0.2500  0.1771                  0.2500
tesla      0.0000  0.0000                  0.0000


In [12]:
query_metrics = results_df.groupby("query").agg(
    hit_rate               = ("hit", "mean"),
    mrr                    = ("reciprocal_rank", "mean"),
    hard_negative_accuracy = ("ranked_above_hard_negative", "mean")
).round(4)

print(query_metrics.to_string())

                                                            hit_rate     mrr  hard_negative_accuracy
query                                                                                               
What concerns were raised during the Q&A?                      0.250  0.1875                   0.250
What did management say about demand trends?                   0.000  0.0000                   0.000
What did the company say about AI investments or strategy?     0.250  0.0833                   0.250
What did the company say about margins or profitability?       0.125  0.0625                   0.125
What did the company say about revenue growth?                 0.250  0.1667                   0.250
What risks did management mention?                             0.250  0.2500                   0.250
What was management's outlook for the next quarter?            0.250  0.1875                   0.250
What were the key highlights from this earnings call?          0.500  0.4375               

In [13]:
failures = results_df[results_df["hit"] == 0][[
    "company", "quarter", "query", "positive_id", "retrieved_ids"
]]

if len(failures) == 0:
    print("No failures — positive chunk retrieved in all cases.")
else:
    print(f"{len(failures)} failure(s) found:\n")
    print(failures.to_string(index=False))

49 failure(s) found:

company quarter                                                      query            positive_id                                                                                                            retrieved_ids
 amazon      Q3             What did the company say about revenue growth? amazon_Q3_2025_chunk_1                                                                                                                       []
 amazon      Q3   What did the company say about margins or profitability? amazon_Q3_2025_chunk_1                                                                                                 [amazon_Q3_2025_chunk_0]
 amazon      Q3               What did management say about demand trends? amazon_Q3_2025_chunk_0                                                                                                                       []
 amazon      Q3 What did the company say about AI investments or strategy? amazon_Q3_2025_chunk_0     